# ACDC V4 — full 3D EPI optimization from a selected plane

Based on **06_acdc_echo32_b0_s0_t2star_MRzeroV3.ipynb**. This version reads the
3D GRE reference, 3D EPI and two-echo 3D B0 acquisitions in `b0map/`.

**Workflow:** load the volumes, inspect the three linked panels, choose X/Y/Z
and a slice, then click **Use this slice**. Only then run the simulation cells.
The selected plane defines the image-loss region. The object, B0, coil fields,
encoding and reconstruction remain **three-dimensional**.

The B0 acquisition was confirmed to have **all ACDC currents at zero**.
Its filename is not used to infer the current state. It contains two echoes,
so V4 does not apply V3's 32-echo relaxation/dispersion fit.

The full-volume point-spin Fourier model uses V3's off-resonance phase convention,
first-order current response, constrained image/field objective and disabled-channel
handling. It is a dedicated PyTorch 3D forward model, rather than a call to the
2D MRzero graph. Its complex signal and current gradients are tested against a
direct 3D calculation. Unresolved subvoxel dephasing, receive-coil phase and a
measured T1/RF steady-state model are not inferred from these three scans.

Run through the viewer first. Running all cells without confirming a slice stops
at the selection check and does not launch a simulation.

In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime
import json, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd() if (Path.cwd()/"Experiment").is_dir() else Path.cwd().parent
for folder in (PROJECT_ROOT/"Experiment", PROJECT_ROOT/"MRzero_validation"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from volume_twix import read_twix_volume
from volume_epi import read_epi3d_sequence
from volume_inputs import build_volume_bundle, prepare_volume_shim, sample_record, with_native_pixel_signs, sample_patient_plane
from volume_slice_viewer import VolumeSliceViewer, extract_volume_slice
from volume_epi_model import VolumeEPIModel
from volume_optimization import optimize_volume_image, export_volume_waveform
from export_acdc_waveform import validate_waveform_settings

INPUT_DIR = PROJECT_ROOT/"b0map"
REFERENCE_PATH = INPUT_DIR/"meas_MID00199_FID50220_acdc3d_reference.dat"
EPI_PATH = INPUT_DIR/"meas_MID00198_FID50219_acdc3d_scan.dat"
B0_PATH = INPUT_DIR/"meas_MID00212_FID50231_gre_b0map_2026_ACDC_CAL_p1_ch0.dat"
SEQUENCE_DIR = INPUT_DIR/"seq"
EPI_SEQUENCE_PATH = SEQUENCE_DIR/"gre_epi3d_128x128x128_fov200x200x200mm_nav3_line_triggers.seq"
CALIBRATION_PATH = PROJECT_ROOT/"data/ACDC_3T.npz"
CACHE_DIR = PROJECT_ROOT/"MRzero_validation/results_volume_twix/cache"

RECEIVER_INDICES = None        # None uses every measured receiver.
INITIAL_AXIS = "z"             # Patient X, Y or Z, not raw array axes.
INITIAL_SLICE = None           # None starts at the middle of the displayed axis.
B0_GLOBAL_OFFSET_HZ = 0.0      # Optional known absolute frequency/alias correction.
# Normally leave these at None. An explicit 4x4 native-to-patient transform is
# available for a separately verified scanner geometry convention.
PATIENT_FROM_NATIVE_M = {"reference": None, "epi": None, "b0": None}

## 1. Reconstruct and register the measured volumes

The readers combine all selected receivers, correct each EPI partition with its
own three navigators, and remove the B0 acquisition's readout, phase and partition
oversampling. Images are resampled using their measured positions and orientations,
not by pairing equal slice indices. Raw files are never modified.

The native Siemens B0 scan needs **no Pulseq file**: echo times, encoding sizes
and geometry are read from its Twix metadata. The EPI Pulseq file in `b0map/seq/`
provides the forward model's acquisition timing and encoding; its signature
is checked against the EPI Twix before simulation.

X/Y/Z in the viewer are **patient coordinates in millimetres**. The native EPI
readout/phase/partition grid is retained separately for the forward calculation.
The 3D reader documents its Siemens PRS-to-RPS convention in the saved metadata;
no anatomical rotation or registration is fitted from the images.

The two-echo map is spatially unwrapped within measured signal components. Its
absolute integer frequency branch cannot be identified from two echoes alone;
`B0_GLOBAL_OFFSET_HZ` permits a known correction. Missing field samples remain
masked, including where one volume does not cover another.

The Siemens B0 image uses the voxel-axis signs defined by `twixtools.geometry`,
while the Pulseq acquisitions retain their declared native gradient mapping.
The actual Pulseq console `OrientationMapping` can differ from the MDH directions;
check the physical alignment in the viewer. This distinction is also documented
in the [Pulseq interpreter discussion](https://github.com/pulseq/pulseq/discussions/285).
No anatomical rotation, translation or sign search is applied automatically.


In [ ]:
records = {}
for name, path, kind in (("reference", REFERENCE_PATH, "reference"),
                         ("epi", EPI_PATH, "epi"), ("b0", B0_PATH, "b0")):
    print(f"Reconstructing {name}: {path.name}", flush=True)
    records[name] = read_twix_volume(
        path, kind, receiver_indices=RECEIVER_INDICES, cache_dir=CACHE_DIR,
        patient_from_native_m=PATIENT_FROM_NATIVE_M[name])
    print("  shape:", records[name]["magnitude"].shape,
          "  cached:", records[name]["metadata"]["cache_hit"])

# Native Siemens Cartesian voxel RPS directions differ from the raw MDH
# gradient-direction triplet. Use twixtools' documented RPS-to-patient convention.
# Pulseq reference/EPI retain their declared native mapping, which is recorded
# separately and can be overridden above if the console OrientationMapping differs.
if PATIENT_FROM_NATIVE_M["b0"] is None:
    records["b0"] = with_native_pixel_signs(
        records["b0"], [-1, -1, -1],
        reason="Siemens Cartesian voxel RPS-to-patient convention from twixtools.geometry (PRS column flips plus RPS slice-axis flip).")


records["b0"]["metadata"].update(
    applied_ACDC_current_known=True, applied_ACDC_currents_A=[0.0]*32,
    current_note="Zero ACDC currents confirmed by the user for this acquisition.")
frequency_adjustment_Hz = (records["b0"]["metadata"]["frequency_Hz"]
                           - records["epi"]["metadata"]["frequency_Hz"])
print("B0-to-EPI reference-frequency adjustment [Hz]:", frequency_adjustment_Hz)
volume_bundle, native_scene = build_volume_bundle(
    records["reference"], records["epi"], records["b0"],
    b0_offset_Hz=frequency_adjustment_Hz+B0_GLOBAL_OFFSET_HZ)
print("Patient-grid viewer shape:", volume_bundle["reference_magnitude"].shape)
print("B0 alias period [Hz]:", records["b0"]["metadata"]["frequency_alias_period_Hz"])

## 2. Select the plane before any simulation

Change **Normal axis** and move the **Slice** slider. The GRE, EPI and B0 panels
always show the same physical plane. Click **Use this slice**, then continue below.
Changing the slider later invalidates the previous confirmation; rerun from the
selection cell to rebuild the optimization mask.

If widgets are unavailable in the notebook kernel, install `ipywidgets` in that
kernel (`%pip install ipywidgets`) and restart it. The full environment is described
in `requirements-experiment.txt`.

In [ ]:
viewer = VolumeSliceViewer(volume_bundle, axis=INITIAL_AXIS, index=INITIAL_SLICE)
display(viewer.widget)

## 3. Freeze the selected plane and set the optimization parameters

A reconstructed slice of a 3D scan is not a separate RF excitation. V4 keeps the
whole slab and optimizes the reference mismatch on a thin patient-aligned ROI.
All supported object voxels contribute to signal formation and field constraints.
The same chronological line waveform is repeated for each acquired kz partition;
V4 does not independently assign a different waveform to every partition.

The fixed calibration anchor below is copied from V3. Keep it fixed when selecting
a different plane. The source calibration's physical placement is still the declared
V3 placement; image browsing does not independently validate its axes or origin.

In [ ]:
selection = viewer.snapshot()  # Requires an explicit click on Use this slice.

def require_same_selection():
    if viewer.snapshot() != selection:
        raise RuntimeError("The selected plane changed. Rerun from the selection cell before simulation or export.")

ROI_THICKNESS_MM = None         # None uses one patient-grid display spacing.
OBJECT_THRESHOLD_FRACTION = 0.04
T2STAR_MS = None                # None: no inferred decay. Or supply a positive constant.

ALIGNMENT_ANGLE_DEG = 90.0
ALIGNMENT_MIRROR_X = False
ALIGNMENT_MIRROR_Y = False
ALIGNMENT_REFERENCE_POSE = {'slice_position_mm': [-2.7755575615628914e-17, 0.5980858206748962, -3.4637004375781544e-10], 'slice_quaternion_wxyz': [0.7070966958999634, -0.0037749139592051506, 0.00040555562009103596, 0.7071066498756409]}
CALIBRATION_PARAMETERS = {
    "units": "Hz/A", "registration_confirmed": True,
    "patient_to_source_m": np.eye(4).tolist(),
    "source_sha256": "bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c"}

DISABLED_CHANNELS = [9]          # Physical outputs 1..32, retained as zero CSV columns.
MAX_CURRENT_A = 3.0
TOTAL_CURRENT_LIMIT_A = 30.0
FIELD_LIMIT_HZ = 1000.0
CURRENT_RESPONSE = {"enabled": True, "tau_s": 133e-6/np.log(100.)}
HARDWARE = {"min_update_interval_s": 8e-6, "dac_resolution_A": 0.0006,
            "max_slew_A_per_s": None, "avg_abs_current_A": None,
            "coil_R_ohm": None, "coil_L_H": None,
            "max_voltage_V": None, "max_coil_power_W": None}
WAVEFORM_PARAMETERS = validate_waveform_settings({
    "channel_map": list(range(1, 33)), "channel_polarity": [-1]*32,
    "disabled_channels": DISABLED_CHANNELS, "max_current_A": MAX_CURRENT_A,
    "total_current_limit_A": TOTAL_CURRENT_LIMIT_A, "min_dac_interval_s": 8e-6}, 32)

GPU_REQUIRED = True             # The full 128-cubed optimization is expensive on CPU.
MODEL_DTYPE = torch.float32
INTERPOLATION_NODES = 8
INTERPOLATION_MAX_NODES = 64
INTERPOLATION_TOLERANCE = 1e-3
IMAGE_INITIALIZATION = "static"
FIELD_GUIDANCE_WEIGHT = 0.05
FIELD_GUIDANCE_SCALE_HZ = 100.0
MAX_ITERATIONS = 300
LEARNING_RATE_A = 0.01
PATIENCE = 60
LIVE_UPDATE_EVERY = 20         # Live figure and archived images, as in V3.
RUN_OPTIMIZATION = True
EXPORT_WAVEFORM = True

normal = "xyz".index(selection["axis"])
display_step_m = float(np.diff(volume_bundle["axes_m"][normal]).min())
roi_thickness_m = display_step_m if ROI_THICKNESS_MM is None else ROI_THICKNESS_MM*1e-3
OUTPUT_DIR = PROJECT_ROOT/"MRzero_validation/results_volume_v4"/datetime.now().strftime("v4_%Y%m%d_%H%M%S_%f")
WAVEFORM_OUTPUT_DIR = INPUT_DIR/"waveform_v4"
print("Confirmed plane:", selection)
print("ROI thickness [mm]:", roi_thickness_m*1000)

## 4. Validate the acquired 3D sequence and sample the coil fields

The Pulseq signature is checked against the EPI Twix. The reader verifies the full
Cartesian kz grid, actual ADC times, identical RF-relative trains across partitions
and one ext1 trigger per acquired imaging line. The model preserves B0 evolution
during the readout; it does not replace ADC times by one time per line.

Object samples without measured B0 or a complete calibration interpolation stencil
are excluded and counted. Their amplitude is not redistributed. Inspect the reported
excluded fraction before interpreting an optimization.

In [ ]:
require_same_selection()
acquisition, sequence_timing = read_epi3d_sequence(
    EPI_SEQUENCE_PATH, expected_signature_md5=records["epi"]["metadata"]["sequence_signature_md5"])
if tuple(acquisition["matrix_size"]) != records["epi"]["magnitude"].shape:
    raise ValueError("EPI raw data and selected 3D sequence have different matrices.")
if not np.allclose(acquisition["FOV_m"], native_scene["FOV_m"], rtol=0, atol=1e-6):
    raise ValueError("EPI raw data and sequence have different fields of view.")
volume = prepare_volume_shim(
    native_scene, selection, CALIBRATION_PATH, CALIBRATION_PARAMETERS,
    alignment_reference_pose=ALIGNMENT_REFERENCE_POSE,
    alignment_angle_deg=ALIGNMENT_ANGLE_DEG,
    alignment_mirror_x=ALIGNMENT_MIRROR_X, alignment_mirror_y=ALIGNMENT_MIRROR_Y,
    roi_thickness_m=roi_thickness_m, object_threshold_fraction=OBJECT_THRESHOLD_FRACTION,
    T2star_s=None if T2STAR_MS is None else T2STAR_MS*1e-3)
print("Acquired matrix:", acquisition["matrix_size"])
print("Partitions / line commands per partition:", sequence_timing["partition_count"], acquisition["ny"])
print("TE / TR [ms]:", acquisition["TE_s"]*1000, acquisition["TR_s"]*1000)
print(json.dumps(volume["report"], indent=2))

# Inspect actual field coverage on the chosen plane before launching a simulation.
coverage, shown = sample_patient_plane(volume["support"].astype(float), native_scene, volume_bundle, selection)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True, layout="constrained")
for ax, values, title in zip(axes, [shown["reference_magnitude"], coverage],
                            ["Selected GRE plane", "Object with measured B0 and coil coverage"]):
    ax.pcolormesh(shown["column_edges_m"]*1000, shown["row_edges_m"]*1000,
                  values, shading="flat", cmap="gray")
    ax.set(title=title, aspect="equal", xlabel=f"Patient {shown['orientation']['column_axis'].upper()} [mm]",
           ylabel=f"Patient {shown['orientation']['row_axis'].upper()} [mm]")
plt.show(); plt.close(fig)


## 5. Construct the full-volume model and check temporal accuracy

This is the first simulation stage. Spatial encoding remains 3D for every choice
of X/Y/Z. The selected plane changes only the fixed image-loss mask.

Chebyshev interpolation reduces the cost of evaluating the field at every ADC
sample. V4 compares increasing node counts on the actual baseline and checks the
final candidate again. Exact ADC-time mode is available in the model for small
validation problems. A failed accuracy check prevents waveform export.

In [ ]:
require_same_selection()
if GPU_REQUIRED and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the configured full-volume run. The measured-volume viewer remains usable without CUDA.")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = VolumeEPIModel(
    volume, acquisition, device=device, dtype=MODEL_DTYPE,
    interpolation_nodes=INTERPOLATION_NODES, checkpoint_lines=True,
    current_response=CURRENT_RESPONSE, max_current_A=WAVEFORM_PARAMETERS["max_current_A"])
model.set_target(volume["reference_magnitude"], volume["roi_mask"], gain=1.0)
zero_currents = np.zeros((model.nc, model.nstates))
print("Device:", model.device, " | full-volume shape:", model.shape, flush=True)
print("Checking temporal interpolation on the full-volume baseline...", flush=True)
baseline_accuracy = model.adapt_interpolation(
    zero_currents, tolerance=INTERPOLATION_TOLERANCE, maximum_nodes=INTERPOLATION_MAX_NODES)
if not baseline_accuracy["passed"]:
    raise RuntimeError("Increase INTERPOLATION_MAX_NODES before optimization.")
with torch.no_grad():
    baseline_image, baseline_kdata = map(model.numpy, model.forward(zero_currents))
print("Baseline selected-plane relative MSE:", float(model.torch_objective(model.tensor(zero_currents))))
print("Temporal plan:", baseline_accuracy)

In [ ]:
def plane_from_native_image(image):
    return sample_patient_plane(np.abs(image), native_scene, volume_bundle, selection)


def show_comparison(simulated, title):
    image, shown = plane_from_native_image(simulated)
    reference = shown["reference_magnitude"]
    measured = shown["epi_magnitude"]
    scale = float(np.nanmax(records["reference"]["magnitude"]))
    # One display-only gain; this never changes the optimizer's fixed target.
    valid = np.isfinite(reference) & np.isfinite(measured)
    gain = np.sum(reference[valid]*measured[valid])/max(np.sum(measured[valid]**2), 1e-30)
    panels = [reference/scale, gain*measured/scale, image]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True, layout="constrained")
    for ax, values, label in zip(axes, panels, ["Measured 3D GRE reference", "Measured 3D EPI", title]):
        ax.pcolormesh(shown["column_edges_m"]*1000, shown["row_edges_m"]*1000,
                      values, shading="flat", cmap="gray", vmin=0, vmax=1)
        ax.set(title=label, aspect="equal", xlabel=f"Patient {shown['orientation']['column_axis'].upper()} [mm]",
               ylabel=f"Patient {shown['orientation']['row_axis'].upper()} [mm]")
    plt.show()
    plt.close(fig)

show_comparison(baseline_image, "Full 3D simulation without ACDC")

## 6. Optimize the acquired volume using the selected plane

The image term compares the full 3D reconstruction to the measured reference on
the selected patient-plane ROI. The residual-field guide covers all supported
3D object samples. Channel limits, the total-current limit and calibrated field
limits are enforced, including the first-order amplifier response and the
switch-off tail. Disabled physical channels remain exactly zero.

`FIELD_GUIDANCE_WEIGHT=0` selects the image term alone. With a positive weight,
the optimizer retains the best **combined objective**, as in V3. The reference,
mask and intensity gain stay fixed during optimization.

The V3 live display is retained: **Reference**, **Baseline**, **Current**, signed
magnitude residual, image-MSE history and channel-current heatmap. It shows the
confirmed patient plane, while simulation and optimization remain fully 3D.
`LIVE_UPDATE_EVERY` controls the display and archive interval (20 by default).
The curve includes every evaluated iteration, and the live figure also updates
at the final evaluated state. The returned solution can be an earlier state
with a lower combined objective; section 7 displays that retained candidate.

A separate slider compares the reference, baseline and periodic iteration images.
Its shared intensity scale and orientation remain fixed. Only these 2D planes are
archived, in `optimization_images.npz` and a standalone `optimization_images.html`.
The HTML file works offline if notebook HTML interaction is unavailable.


In [ ]:
from live_optimization import LiveOptimizationPlot
from optimization_image_history import OptimizationImageHistory

require_same_selection()
solution = None
image_history = None
live = None
image_history_npz = OUTPUT_DIR/"optimization_images.npz"
image_history_html = OUTPUT_DIR/"optimization_images.html"

if RUN_OPTIMIZATION:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # Sample the optimizer's fixed, normalized target on the confirmed patient plane.
    # No image-dependent gain, transposition or rotation is fitted for this display.
    reference_plane, shown = plane_from_native_image(volume["reference_magnitude"])
    baseline_plane, _ = plane_from_native_image(baseline_image)
    roi_plane, _ = sample_patient_plane(
        volume["roi_mask"].astype(float), native_scene, volume_bundle, selection)
    display_valid = np.isfinite(reference_plane) & np.isfinite(baseline_plane)
    display_roi = display_valid & np.isfinite(roi_plane) & (roi_plane >= 0.5)

    def display_plane(values):
        return np.where(display_valid & np.isfinite(values), values, 0.)

    reference_plane = display_plane(reference_plane)
    baseline_plane = display_plane(baseline_plane)
    extent_mm = np.array([shown["column_edges_m"][0], shown["column_edges_m"][-1],
                          shown["row_edges_m"][0], shown["row_edges_m"][-1]])*1000
    plane_label = f"Patient {selection['axis'].upper()} = {selection['coordinate_m']*1000:.2f} mm"
    image_history = OptimizationImageHistory(
        reference_plane, baseline_plane, every=LIVE_UPDATE_EVERY,
        title=f"Current scan — optimization images | {plane_label}", origin="lower", language="en")
    live = LiveOptimizationPlot(
        reference=reference_plane, baseline=baseline_plane, current_limit_a=MAX_CURRENT_A,
        extent_mm=extent_mm, roi_mask=display_roi if display_roi.any() else None,
        title=f"V4: acquired 3D EPI | {plane_label}")
    for ax in live.figure.axes[:4]:
        ax.set(xlabel=f"Patient {shown['orientation']['column_axis'].upper()} [mm]",
               ylabel=f"Patient {shown['orientation']['row_axis'].upper()} [mm]")

    def progress(iteration, snapshot, currents):
        require_same_selection()
        # This image is reused from the objective evaluation; no extra 3D simulation.
        current_plane, _ = plane_from_native_image(snapshot["image"])
        current_plane = display_plane(current_plane)
        live.set_history(snapshot["history"]["iteration"], snapshot["history"]["image_mse"])
        live.update(iteration=iteration, image=current_plane, currents_a=currents,
                    loss=snapshot["image_mse"], nrmse_percent=100*np.sqrt(snapshot["image_mse"]),
                    max_field_hz=snapshot["max_field_Hz"], field_limit_hz=FIELD_LIMIT_HZ)
        if image_history.record({"iteration": iteration, "image": current_plane}):
            image_history.save_npz(image_history_npz)
            image_history.write_html(image_history_html)
            image_history.display()

    try:
        solution = optimize_volume_image(
            model, initialization=IMAGE_INITIALIZATION,
            total_current_limit_A=TOTAL_CURRENT_LIMIT_A, field_limit_Hz=FIELD_LIMIT_HZ,
            field_guidance_weight=FIELD_GUIDANCE_WEIGHT, field_guidance_scale_Hz=FIELD_GUIDANCE_SCALE_HZ,
            max_iterations=MAX_ITERATIONS, learning_rate_A=LEARNING_RATE_A, patience=PATIENCE,
            hardware=HARDWARE, callback=progress, callback_every=LIVE_UPDATE_EVERY)
    finally:
        status = "Completed — last evaluated state" if solution is not None else "Stopped — last evaluated state"
        live.finish(status)
        live.figure.savefig(OUTPUT_DIR/"optimization_progress.png", dpi=140)
    print("Stop reason:", solution["stop_reason"])
    print("Initial / best / quantized total objective:", solution["initial_objective"],
          solution["best_objective"], solution["quantized_objective"])
    print("Optimization image slider:", image_history_html)
else:
    print("Optimization is disabled; the baseline simulation remains available.")


## 7. Check the final candidate and inspect the selected plane

The final commands are checked against a finer temporal plan. If this fails,
increase the interpolation settings and rerun the model/optimization cells.
The hardware and objective checks are also repeated after nominal DAC quantization.

In [ ]:
require_same_selection()
final_accuracy = None
optimized_image = optimized_kdata = None
if solution is not None:
    comparison = model.convergence_check(
        solution["currents_A"], refined_nodes=min(model.nx, 2*model.interpolation_nodes))
    final_accuracy = dict(comparison, tolerance=INTERPOLATION_TOLERANCE,
        passed=comparison["relative_complex_kdata_error"] <= INTERPOLATION_TOLERANCE)
    quantized_comparison = model.convergence_check(
        solution["quantized_currents_A"], refined_nodes=min(model.nx, 2*model.interpolation_nodes))
    final_accuracy["quantized_check"] = quantized_comparison
    final_accuracy["passed"] &= quantized_comparison["relative_complex_kdata_error"] <= INTERPOLATION_TOLERANCE
    with torch.no_grad():
        optimized_image, optimized_kdata = map(model.numpy, model.forward(solution["quantized_currents_A"]))
    print("Final interpolation check:", final_accuracy)
    print("Continuous hardware limits:", solution["hardware_after_DAC"]["all_limits_met"])
    show_comparison(optimized_image, "Full 3D simulation with ACDC (nominal DAC)")
    history = solution["history"]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot([r["iteration"] for r in history], [r["objective"] for r in history], label="Combined objective")
    ax.plot([r["iteration"] for r in history], [r["image_mse"] for r in history], label="Image relative MSE")
    ax.set(xlabel="Iteration", ylabel="Normalized objective")
    ax.legend(); plt.show(); plt.close(fig)

## 8. Save the scene, selection, optimization and provenance

Results use a new timestamped V4 folder. Raw scans, V3 and existing waveforms are
not overwritten. The saved selection is a patient-plane ROI, while saved images
and k-space remain full 3D volumes. Source hashes, geometry conventions, calibration
placement, sequence identity, interpolation checks and hardware reports are retained.

In [ ]:
def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray): return json_safe(value.tolist())
    if isinstance(value, np.generic): return json_safe(value.item())
    if isinstance(value, Path): return str(value)
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

require_same_selection()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = dict(
    notebook="06_acdc_echo32_b0_s0_t2star_MRzeroV4 (3D).ipynb", based_on="06_acdc_echo32_b0_s0_t2star_MRzeroV3.ipynb",
    selection=selection, preparation=volume["report"], sequence=sequence_timing,
    measured_volumes={name: record["metadata"] for name, record in records.items()},
    native_center_m=native_scene["center_m"], native_basis_patient=native_scene["basis_patient"],
    b0_reference_adjustment_Hz=frequency_adjustment_Hz, b0_global_offset_Hz=B0_GLOBAL_OFFSET_HZ,
    model=model.interpolation_report, baseline_accuracy=baseline_accuracy,
    final_accuracy=final_accuracy, waveform_parameters=WAVEFORM_PARAMETERS,
    optimization=solution,
    image_history=None if image_history is None else dict(
        every=image_history.every, iterations=image_history.arrays()["iterations"],
        selection=selection, origin=image_history.origin, extent_mm=extent_mm,
        archive=image_history_npz.name, viewer=image_history_html.name,
        live_figure="optimization_progress.png"),
    no_hardware_action_performed=True)
arrays = dict(reference_magnitude=volume["reference_magnitude"], measured_epi_magnitude=native_scene["epi_magnitude"],
              measured_b0_Hz=native_scene["b0_Hz"], object_support=volume["support"], roi_mask=volume["roi_mask"],
              baseline_image=baseline_image, baseline_kdata=baseline_kdata,
              native_center_m=native_scene["center_m"], native_basis_patient=native_scene["basis_patient"])
if solution is not None:
    arrays.update(currents_A=solution["currents_A"], quantized_currents_A=solution["quantized_currents_A"],
                  optimized_image=optimized_image, optimized_kdata=optimized_kdata)
np.savez_compressed(OUTPUT_DIR/"volume_optimization.npz", **arrays)
(OUTPUT_DIR/"volume_optimization.json").write_text(
    json.dumps(json_safe(summary), indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
print("Saved:", OUTPUT_DIR)

## 9. Export the full-acquisition waveform

The timed CSV repeats the optimized line commands across every acquired partition
and explicitly returns to zero after each train, before the next RF/navigators.
Timing comes from the matching 3D Pulseq sequence. A sidecar links the waveform to
the selected plane, full-volume scene, current limits and source hashes.

**This 3D export is a timed CSV.** Replacing every time by zero would make the
inter-partition off events consume imaging triggers and shift subsequent commands.
V4 therefore does not generate an equivalent `_zero_time.csv`. The waveform player
must support the timed event schedule from the first imaging trigger; this notebook
only writes files and does not configure or start the controller.

Export requires a finite objective improvement that survives quantization, passing
hardware checks and a passing final temporal-accuracy check.

In [ ]:
require_same_selection()
waveform_export = None
if EXPORT_WAVEFORM and solution is not None:
    waveform_export = export_volume_waveform(
        solution, sequence_timing, WAVEFORM_PARAMETERS, WAVEFORM_OUTPUT_DIR,
        metadata={"selection": selection, "preparation": volume["report"],
                  "source_sha256": {name: record["metadata"]["source_sha256"] for name, record in records.items()},
                  "native_center_m": native_scene["center_m"],
                  "native_basis_patient": native_scene["basis_patient"]},
        time_interpolation_report=final_accuracy)
    print("Full 3D timed CSV:", waveform_export["csv"]["csv_file"])
    print("Scene/timing sidecar:", waveform_export["sidecar_json"])
else:
    print("No waveform exported: export or optimization is disabled.")
summary["waveform_export"] = waveform_export
(OUTPUT_DIR/"volume_optimization.json").write_text(
    json.dumps(json_safe(summary), indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")